In [1]:
import importlib, subprocess, sys, os, warnings
if importlib.util.find_spec('xgboost') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'xgboost', '-q'])

import numpy as np
import pandas as pd
import xgboost as xgb
import joblib
from sklearn.metrics import r2_score, mean_squared_error
from importlib.util import find_spec

warnings.filterwarnings('ignore')
os.makedirs('./models', exist_ok=True)

DATA_PATH = './data_sets.xlsx'

# 6-season calendar labels in English
SEASON_NAME = {
     1: 'Winter',      2: 'Winter',
     3: 'Spring',      4: 'Spring',
     5: 'Summer',      6: 'Summer',
     7: 'Monsoon',     8: 'Monsoon',
     9: 'Autumn',     10: 'Autumn',
    11: 'Pre-Winter', 12: 'Pre-Winter',
}
SEASON_ENC = {
    'Winter': 0, 'Spring': 1, 'Summer': 2,
    'Monsoon': 3, 'Autumn': 4, 'Pre-Winter': 5,
}
DISC_ENC  = {'None': 0, 'Low': 1, 'Medium': 2, 'High': 3}
DISC_RATE = {'None': 0.0, 'Low': 0.03, 'Medium': 0.05, 'High': 0.10}

In [2]:
# Load master dataset
df = pd.read_excel(DATA_PATH, sheet_name='Monthly_Data')
df['Discount Band'] = df['Discount Band'].fillna('None')

ALL_PRODUCTS  = sorted(df['Product'].unique())
ALL_SEGMENTS  = sorted(df['Segment'].unique())
BASE_YEAR     = int(df['Year'].min())
SEG_ENC       = {s: i for i, s in enumerate(ALL_SEGMENTS)}
PROD_MAP      = {p: i for i, p in enumerate(ALL_PRODUCTS)}
ACTIVE_MONTHS = df.groupby('Product')['Month Number'].apply(lambda x: sorted(x.unique().tolist())).to_dict()
PRODUCT_SPECS = df.sort_values(['Year','Month Number']).groupby('Product').last()[['Manufacturing Price','Sale Price','Segment']].to_dict('index')
PRODUCT_DISC  = df.groupby('Product')['Discount Band'].agg(lambda x: x.mode()[0]).to_dict()

# Feature engineering
df['Date']         = pd.to_datetime(df['Year'].astype(str) + '-' + df['Month Number'].astype(str).str.zfill(2) + '-01')
df                 = df.sort_values(['Product', 'Date']).reset_index(drop=True)
df['Month']        = df['Date'].dt.month
df['Year_num']     = df['Date'].dt.year
df['Quarter']      = df['Date'].dt.quarter
df['Month_sin']    = np.sin(2 * np.pi * df['Month'] / 12)
df['Month_cos']    = np.cos(2 * np.pi * df['Month'] / 12)
df['Qtr_sin']      = np.sin(2 * np.pi * df['Quarter'] / 4)
df['Qtr_cos']      = np.cos(2 * np.pi * df['Quarter'] / 4)
df['Season_enc']   = df['Month'].map(SEASON_NAME).map(SEASON_ENC)
df['Year_trend']   = df['Year_num'] - BASE_YEAR
df['Segment_enc']  = df['Segment'].map(SEG_ENC)
df['Product_enc']  = df['Product'].map(PROD_MAP)
df['DiscBand_enc'] = df['Discount Band'].map(DISC_ENC).fillna(0).astype(int)

for lag in [1, 2, 3, 6, 12]:
    df[f'Sales_lag{lag}'] = df.groupby('Product')['Sales'].shift(lag)

for win in [3, 6]:
    df[f'Sales_roll{win}_mean'] = df.groupby('Product')['Sales'].transform(lambda x: x.shift(1).rolling(win, min_periods=1).mean())
    df[f'Sales_roll{win}_std']  = df.groupby('Product')['Sales'].transform(lambda x: x.shift(1).rolling(win, min_periods=2).std().fillna(0))

df['Discount_Rate']      = np.where(df['Gross Sales'] != 0, df['Discounts'] / df['Gross Sales'], 0)
df['Revenue_per_Unit']   = np.where(df['Units Sold']  != 0, df['Sales'] / df['Units Sold'], 0)
df['Cost_per_Unit']      = np.where(df['Units Sold']  != 0, df['COGS']  / df['Units Sold'], 0)
df['Price_to_ManufCost'] = np.where(df['Manufacturing Price'] != 0, df['Sale Price'] / df['Manufacturing Price'], 0)

DROP         = ['Sales', 'Profit', 'Date', 'Segment', 'Product', 'Discount Band', 'Month Number', 'Year']
FEATURE_COLS = [c for c in df.columns if c not in DROP]
LAG_COLS     = [c for c in FEATURE_COLS if 'lag' in c or 'roll' in c]

df_model = df.dropna(subset=LAG_COLS).reset_index(drop=True)
split    = int(len(df_model) * 0.80)
X_train, y_train = df_model.iloc[:split][FEATURE_COLS], df_model.iloc[:split]['Sales']
X_test,  y_test  = df_model.iloc[split:][FEATURE_COLS], df_model.iloc[split:]['Sales']

print(f'Data     : {len(df):,} rows | {len(ALL_PRODUCTS)} products | {BASE_YEAR}–{int(df["Year_num"].max())}')
print(f'Features : {len(FEATURE_COLS)} | Train: {len(X_train):,} | Test: {len(X_test):,}')

Data     : 2,166 rows | 31 products | 2015–2026
Features : 31 | Train: 1,435 | Test: 359


In [3]:
# Train XGBoost Sales model
print('Training...')
model = xgb.XGBRegressor(
    n_estimators=800,   learning_rate=0.03,  max_depth=6,
    subsample=0.80,     colsample_bytree=0.80,
    reg_alpha=0.1,      reg_lambda=1.0,
    min_child_weight=5, gamma=0.1,
    random_state=42,    tree_method='hist',
    early_stopping_rounds=40, eval_metric='rmse', verbosity=0,
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

preds = model.predict(X_test)
r2    = r2_score(y_test, preds)
mape  = np.mean(np.abs((y_test - preds) / np.where(y_test == 0, 1, y_test))) * 100
rmse  = np.sqrt(mean_squared_error(y_test, preds))
grade = 'Excellent' if r2 > 0.95 else 'Good' if r2 > 0.85 else 'Needs review'

print(f'R2   = {r2:.4f}  ({grade})')
print(f'MAPE = {mape:.2f}%')
print(f'RMSE = {rmse:,.0f}')

# Save model
model.save_model('./models/sales_model.json')
joblib.dump(model, './models/sales_model.pkl')

# Save all metadata needed by predict.ipynb
joblib.dump({
    'feature_cols':  FEATURE_COLS,
    'prod_map':      PROD_MAP,
    'seg_enc':       SEG_ENC,
    'active_months': ACTIVE_MONTHS,
    'product_specs': PRODUCT_SPECS,
    'product_disc':  PRODUCT_DISC,
    'base_year':     BASE_YEAR,
    'season_name':   SEASON_NAME,
    'season_enc':    SEASON_ENC,
    'all_products':  ALL_PRODUCTS,
    'disc_enc':      DISC_ENC,
    'disc_rate':     DISC_RATE,
}, './models/metadata.pkl')

print()
print('Model saved → models/sales_model.pkl')
print('Open predict.ipynb for all future predictions.')

Training...
R2   = 0.9960  (Excellent)
MAPE = 2.09%
RMSE = 9,610

Model saved → models/sales_model.pkl
Open predict.ipynb for all future predictions.
